### Interactive Notebook UI – How to use this demo

This section provides a simple user interface for our **Music Mood Classifier** directly inside the notebook, so you can run and inspect the model without starting a separate Streamlit app.

**1. Manual feature entry (tab “Manual entry”)**

- Use the sliders to set the Spotify audio features for a single track  
  (e.g. danceability, energy, valence, tempo).  
- Click **“Predict genre”** to run the trained model on this configuration.  
- The notebook will display:
  - the **predicted genre** for that track  
  - the **top 5 predicted genres with probabilities**, to show model confidence.

**2. Batch CSV prediction (tab “Batch CSV”)**

- Click **“Browse” / “Choose file”** and upload a CSV containing one row per track.  
  The file should contain the same feature columns used in training  
  (missing columns are filled with zeros).  
- Click **“Run batch prediction”**.  
- The notebook will show:
  - a preview of the output DataFrame with a new `predicted_genre` column  
  - (optionally) a `confidence` column with the highest class probability.

This UI uses the **same trained model and preprocessing pipeline** as our Streamlit apps, so predictions here are identical to what you see in the web UI.


In [1]:
!pip install lightgbm==4.6.0
!pip install ipywidgets==8.1.2

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import ipywidgets as widgets
from IPython.display import display

slider = widgets.IntSlider(description="Test")
display(slider)

IntSlider(value=0, description='Test')

In [3]:
import sys
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Notebook is in Group_Project/notebooks → project root is one level up
PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT:", PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))

# Import your custom transformer so unpickling works
from src.feature_engineering import MusicFeatureEngineer  # must match your file

MODEL_PATH = PROJECT_ROOT / "models" / "final_model.pkl"
PIPELINE_PATH = PROJECT_ROOT / "models" / "preprocessor.pkl"
ENCODER_PATH = PROJECT_ROOT / "models" / "label_encoder.pkl"


PROJECT_ROOT: C:\Users\l2-lett\OneDrive - UWE Bristol\Group_Project


In [4]:
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

with open(PIPELINE_PATH, "rb") as f:
    pipeline = pickle.load(f)

with open(ENCODER_PATH, "rb") as f:
    le = pickle.load(f)

print("Loaded model, pipeline, and label encoder.")
print("Classes:", list(le.classes_))

Loaded model, pipeline, and label encoder.
Classes: ['acoustic', 'alternative', 'dance', 'electronic', 'heavy', 'vocal']


C:\ProgramData\Anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\ProgramData\Anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\ProgramData\Anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.5.1. This might lead to breaking code or

In [5]:
# Audio feature metadata: (display name, min, max, default, step, help_text)
FEATURE_CONFIG = {
    "popularity":        ("Popularity",        0,   100,  50,    1,   "Spotify popularity score (0=obscure, 100=viral)"),
    "duration_ms":       ("Duration (ms)",     0, 600000, 210000, 1000, "Track length in milliseconds"),
    "explicit":          ("Explicit",          0,    1,   0,     1,   "1 if explicit lyrics, 0 otherwise"),
    "danceability":      ("Danceability",      0.0, 1.0,  0.6, 0.01, "How suitable for dancing (0=low, 1=high)"),
    "energy":            ("Energy",            0.0, 1.0,  0.7, 0.01, "Perceptual intensity (0=calm, 1=intense)"),
    "key":               ("Key",               0,   11,   5,    1,   "Musical key (0=C, 1=C#, ..., 11=B)"),
    "loudness":          ("Loudness (dB)",    -60.0, 0.0, -7.0, 0.1, "Average loudness in decibels"),
    "mode":              ("Mode",              0,    1,   1,    1,   "1=major, 0=minor"),
    "speechiness":       ("Speechiness",      0.0, 1.0,  0.05, 0.01, "Presence of spoken words"),
    "acousticness":      ("Acousticness",     0.0, 1.0,  0.2, 0.01, "Confidence the track is acoustic"),
    "instrumentalness":  ("Instrumentalness", 0.0, 1.0,  0.0, 0.01, "Probability of no vocals"),
    "liveness":          ("Liveness",         0.0, 1.0,  0.12, 0.01, "Probability of live recording"),
    "valence":           ("Valence",          0.0, 1.0,  0.5, 0.01, "Musical positiveness (0=sad, 1=happy)"),
    "tempo":             ("Tempo (BPM)",       0.0, 250.0, 120.0, 0.5, "Estimated beats per minute"),
    "time_signature":    ("Time Signature",    1,    7,   4,    1,   "Beats per bar"),
}

In [6]:
import urllib.parse

PLAYLISTS = {
    "acoustic": [
        {"name": "Peaceful Piano",       "desc": "Soft instrumentals for quiet moments",       "query": "peaceful piano ambient playlist"},
        {"name": "Coffee Shop Acoustic", "desc": "Gentle singer-songwriter for slow mornings", "query": "coffee shop acoustic playlist"},
        {"name": "Deep Focus",           "desc": "Ambient sounds to help you concentrate",     "query": "deep focus study ambient playlist"},
    ],
    "alternative": [
        {"name": "Indie Essentials", "desc": "The best of indie rock and dream pop",  "query": "indie rock essentials playlist"},
        {"name": "90s Alt Rock",     "desc": "Grunge and alternative classics",       "query": "90s alternative grunge rock playlist"},
        {"name": "Road Trip Rock",   "desc": "Guitar-driven tracks for the open road","query": "road trip rock driving playlist"},
    ],
    "dance": [
        {"name": "Today's Top Hits", "desc": "The biggest pop tracks right now",      "query": "todays top hits pop playlist"},
        {"name": "Summer Vibes",     "desc": "Sun-drenched latin pop and tropical",  "query": "summer latin pop vibes playlist"},
        {"name": "Party Anthems",    "desc": "High-energy dance floor bangers",      "query": "party anthems dance floor playlist"},
    ],
    "electronic": [
        {"name": "Electronic Focus", "desc": "Deep house and ambient for focus",     "query": "electronic focus deep house playlist"},
        {"name": "Late Night EDM",   "desc": "Festival-ready drops and big room",    "query": "late night edm festival playlist"},
        {"name": "Synthwave Drive",  "desc": "Retro-future synths for night drives", "query": "synthwave retrowave night drive playlist"},
    ],
    "heavy": [
        {"name": "Metal Classics", "desc": "The definitive heavy metal hall of fame","query": "metal classics rock playlist"},
        {"name": "Gym Fuel",       "desc": "Hard rock and metal for workouts",       "query": "gym workout metal rock playlist"},
        {"name": "Punk Rush",      "desc": "Fast, loud, and raw punk energy",        "query": "punk rock energy playlist"},
    ],
    "vocal": [
        {"name": "Hip-Hop Essentials", "desc": "Tracks that defined hip-hop",        "query": "hip hop rap essentials playlist"},
        {"name": "Lofi Hip-Hop",       "desc": "Lo-fi beats and laid-back flows",    "query": "lofi hip hop chill beats playlist"},
        {"name": "R&B Mood",           "desc": "Smooth R&B and soul for evenings",   "query": "rnb soul mood playlist"},
    ],
}

In [7]:
import warnings
def predict_single(record: dict):
    df = pd.DataFrame([record])
    X_scaled = pipeline.transform(df)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        label_int = model.predict(X_scaled)[0]
        proba = model.predict_proba(X_scaled)[0]
    genre = le.inverse_transform([label_int])[0]
    proba_series = pd.Series(proba, index=le.classes_).sort_values(ascending=False)
    return genre, proba_series

def predict_batch(df_raw: pd.DataFrame) -> pd.DataFrame:
    feature_cols = list(FEATURE_CONFIG.keys())
    for col in feature_cols:
        if col not in df_raw.columns:
            df_raw[col] = 0
    X = df_raw[feature_cols].copy()
    X_scaled = pipeline.transform(X)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        y_pred = model.predict(X_scaled)
        proba = model.predict_proba(X_scaled)

    labels = le.inverse_transform(y_pred)
    result = df_raw.copy()
    result["predicted_genre"] = labels
    result["confidence"] = (proba.max(axis=1) * 100).round(1)
    return result

In [8]:
# Create widgets for each feature (3 columns grid)
sliders = {}
for key, (label, lo, hi, default, step, help_text) in FEATURE_CONFIG.items():
    if isinstance(step, int):
        w = widgets.IntSlider(
            description=label,
            min=int(lo),
            max=int(hi),
            value=int(default),
            step=int(step),
            continuous_update=False,
        )
    else:
        w = widgets.FloatSlider(
            description=label,
            min=float(lo),
            max=float(hi),
            value=float(default),
            step=float(step),
            continuous_update=False,
        )
    sliders[key] = w

button_predict = widgets.Button(description="Predict genre", button_style="success")
output_single = widgets.Output()

def on_predict_click(_):
    with output_single:
        output_single.clear_output()
        record = {k: w.value for k, w in sliders.items()}
        genre, proba_series = predict_single(record)

        print(f"Predicted genre: {genre}")
        if proba_series is not None:
            display(proba_series.head(5).to_frame("probability"))

        # Show playlist suggestions for this genre if we have them
        if genre in PLAYLISTS:
            print("\nSuggested playlists:")
            for pl in PLAYLISTS[genre]:
                url = "https://open.spotify.com/search/" + urllib.parse.quote(pl["query"])
                print(f"- {pl['name']}: {pl['desc']}")
                print(f"  {url}")

button_predict.on_click(on_predict_click)

grid = widgets.GridBox(
    list(sliders.values()),
    layout=widgets.Layout(
        grid_template_columns="repeat(3, 300px)",
        grid_gap="8px 16px",
    ),
)

manual_ui = widgets.VBox(
    [
        widgets.HTML("<h3>Music Mood Classifier – Manual Features</h3>"),
        grid,
        button_predict,
        output_single,
    ]
)

In [9]:
import ipywidgets as widgets
from IPython.display import display

data_dir = PROJECT_ROOT / "data"
csv_files = [f.name for f in data_dir.glob("*.csv")]

options = ["-- choose a CSV --"] + csv_files
dropdown_csv = widgets.Dropdown(
    options=options,
    value="-- choose a CSV --",
    description="CSV file:",
)

button_batch = widgets.Button(description="Run batch prediction", button_style="info")
output_batch = widgets.Output()

def on_batch_click(_):
    with output_batch:
        output_batch.clear_output()
        if dropdown_csv.value == "-- choose a CSV --":
            print("Please select a CSV file from the dropdown.")
            return
        path = data_dir / dropdown_csv.value
        print(f"Reading {path}")
        df = pd.read_csv(path)
        print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
        result = predict_batch(df)
        display(result.head())

button_batch.on_click(on_batch_click)

batch_ui = widgets.VBox(
    [
        widgets.HTML("<h3>Batch CSV Prediction</h3>"),
        dropdown_csv,
        button_batch,
        output_batch,
    ]
)


In [10]:
tabs = widgets.Tab(children=[manual_ui, batch_ui])
tabs.set_title(0, "Manual entry")
tabs.set_title(1, "Batch CSV")

display(tabs)